In [16]:
import os
import re
import requests
import gzip
import torch
import numpy as np
from warcio.archiveiterator import ArchiveIterator
from trafilatura import extract
from langdetect import detect, DetectorFactory
from ftfy import fix_text
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from torch.utils.data import DataLoader, Dataset
import pytorch_lightning as pl
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

# Фиксация сида для воспроизводимости langdetect
DetectorFactory.seed = 42

class CommonCrawlDataModule(pl.LightningDataModule):
    def __init__(self, warc_url, raw_dir='data/raw', processed_dir='data/processed', batch_size=4):
        super().__init__()
        # 1. Определяем устройство: если есть CUDA, используем её
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")

        self.warc_url = warc_url
        self.raw_dir = raw_dir
        self.processed_dir = processed_dir
        self.batch_size = batch_size

        self.tokenizer_gpt2 = GPT2TokenizerFast.from_pretrained("gpt2")

        # 2. Переносим модель на GPU сразу при загрузке
        self.model_gpt2 = GPT2LMHeadModel.from_pretrained("gpt2").to(self.device)
        self.model_gpt2.eval()

        os.makedirs(raw_dir, exist_ok=True)
        os.makedirs(processed_dir, exist_ok=True)

    ## 1 Загрузка и конвертация
    def download_and_extract(self):
        warc_path = os.path.join(self.raw_dir, "sample.warc.gz")

        print(f"Downloading from {self.warc_url}...")
        response = requests.get(self.warc_url, stream=True)

        # ПРОВЕРКА Если статус не 200, значит S3 вернул ошибку (XML)
        if response.status_code != 200:
            print(f"Error: Could not download file. Status code: {response.status_code}")
            print("Server response:", response.text) # Тут вы увидите причину ошибки
            return [] # Возвращаем пустой список, так как загрузка не удалась

        with open(warc_path, "wb") as f:
            # Используем response.content для файлов небольшого размера, или iter_content для больших
            # Если файл большой, то нужно итерироваться
            f.write(response.content)

        texts = []
        # Открываем файл как gzip, так как .gz в расширении
        try:
            with gzip.open(warc_path, 'rb') as stream:
                for record in ArchiveIterator(stream):
                    if record.rec_type == 'response':
                        # Извлекаем только текст, игнорируя HTML разметку и хедеры
                        html_content = record.content_stream().read()
                        content = extract(html_content) # trafilatura отлично чистит HTML
                        if content:
                            texts.append(content)
        except gzip.BadGzipFile:
            print(f"Error: {warc_path} is not a valid gzip file. It might be an HTML error page or an invalid file.")
            try:
                with open(warc_path, 'rb') as f:
                    first_bytes = f.read(200)
                    print(f"First 200 bytes of the file: {first_bytes}")
            except Exception as read_err:
                print(f"Could not read first bytes of file: {read_err}")
            return []
        except Exception as e:
            print(f"An error occurred while processing WARC file: {e}")
            return []

        return texts

    ## 2 Очистка и фильтрация
    def clean_text(self, text):
        # 1. Нормализация Unicode и исправление "битых" символов
        text = fix_text(text)

        # 2. Удаление неизвестных языков (оставляем только 'en' или 'ru')
        try:
            if detect(text) not in ['en', 'ru']:
                return None
        except:
            return None

        # 3. Нормализация пробелов и удаление пустых строк
        text = re.sub(r'\s+', ' ', text).strip()
        if not text:
            return None

        return text

    def segment_text(self, text, max_tokens=800):
        # Разбиение длинных текстов на части (~512-1024 токенов)
        words = text.split()
        segments = [" ".join(words[i:i + max_tokens]) for i in range(0, len(words), max_tokens)]
        return segments

    ## -3: Энтропия и плотность
    @torch.no_grad()
    def calculate_metrics(self, text):
        # Токенизируем текст
        inputs = self.tokenizer_gpt2(text, return_tensors="pt", truncation=True, max_length=1024)

        # ПЕРЕНОСИМ ДАННЫЕ НА GPU
        # Перебираем все ключи в словаре (input_ids, attention_mask) и кидаем на то же устройство, что и модель
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        # Теперь вычисления на видеокарте
        outputs = self.model_gpt2(**inputs, labels=inputs["input_ids"])
        loss = outputs.loss.item()

        density = loss / len(text) if len(text) > 0 else 0
        return loss, density

    def prepare_data(self):
        # tqdm для визуализации прогресса, чтобы не гадать, завис код или нет
        from tqdm import tqdm

        raw_texts = self.download_and_extract()
        processed_data = []

        if not raw_texts:
            print("No raw texts to process. Exiting prepare_data.")
            return

        print(f"Processing {len(raw_texts)} records...")
        for raw_text in tqdm(raw_texts): #прогресс-бар
            cleaned = self.clean_text(raw_text)
            if cleaned:
                segments = self.segment_text(cleaned)
                for seg in segments:
                    entropy, density = self.calculate_metrics(seg)
                    if 3.0 < entropy < 7.0:
                        processed_data.append({"text": seg, "entropy": entropy, "density": density})


        print("Processing and filtering texts...")
        for raw_text in raw_texts:
            cleaned = self.clean_text(raw_text)
            if cleaned:
                segments = self.segment_text(cleaned)
                for seg in segments:
                    entropy, density = self.calculate_metrics(seg)
                    # Фильтрация по энтропии (удаляем слишком предсказуемый или хаотичный текст)
                    if 3.0 < entropy < 7.0:
                        processed_data.append({
                            "text": seg,
                            "entropy": entropy,
                            "density": density
                        })

        # Удаление дубликатов по тексту
        unique_data = {d['text']: d for d in processed_data}.values()
        self.final_data = list(unique_data)

        if not self.final_data:
            print("No data after processing and filtering.")
            return

        # Оценка информационной плотности датасета (взвешенное среднее)
        total_len = sum(len(d['text']) for d in self.final_data)
        avg_density = sum(d['density'] * len(d['text']) for d in self.final_data) / total_len
        print(f"Dataset average information density: {avg_density:.6f}")

    def setup(self, stage=None):
        # Здесь обычно происходит разделение на train/val
        pass

WARC_URL = "https://data.commoncrawl.org/crawl-data/CC-NEWS/2025/02/CC-NEWS-20250201012811-00559.warc.gz"


dm = CommonCrawlDataModule(WARC_URL)
dm.prepare_data()

Using device: cuda


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ERROR:trafilatura.utils:lxml parsing failed: Document is empty
ERROR:trafilatura.utils:lxml parser bytestring Document is empty
ERROR:trafilatura.core:empty HTML tree: None
ERROR:trafilatura.utils:lxml parsing failed: Document is empty
ERROR:trafilatura.utils:lxml parser bytestring Document is empty
ERROR:trafilatura.core:empty HTML tree: None
ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None
ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None
ERROR:trafilatura.utils:lxml parsing failed: Document is empty
ERROR:trafilatura.utils:lxml parser bytestring Document is empty
ERROR:trafilatura.core:empty HTML tree: None


Processing 18083 records...


100%|██████████| 18083/18083 [10:49<00:00, 27.85it/s]


Processing and filtering texts...
Dataset average information density: 0.001459


In [19]:
def run_tokenization_tasks(data):
    texts = [d['text'] for d in data]
    sample_text = texts[0] if texts else "Sample text for tokenization."

    print("\n--- Tokenization Tasks ---")

    # 1. Посимвольная токенизация
    char_vocab = sorted(list(set("".join(texts))))
    char_to_id = {char: i for i, char in enumerate(char_vocab)}
    char_encoded = [char_to_id[c] for c in sample_text if c in char_to_id]
    print(f"Char Vocab Size: {len(char_vocab)}")
    print(f"Sample Char Sequence Length: {len(char_encoded)}")

    # 2. Пословная токенизация
    word_vocab = set()
    for t in texts[:100]: # Ограничение для экономии памяти
        word_vocab.update(re.findall(r'\w+', t.lower()))
    word_to_id = {word: i for i, word in enumerate(word_vocab)}
    word_encoded = [word_to_id[w] for w in re.findall(r'\w+', sample_text.lower()) if w in word_to_id]
    print(f"Word Vocab Size (subset): {len(word_vocab)}")
    print(f"Sample Word Sequence Length: {len(word_encoded)}")

    # 3. BPE Токенизация (Byte Pair Encoding)
    # Используем библиотеку tokenizers от Hugging Face
    bpe_tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
    trainer = trainers.BpeTrainer(vocab_size=5000, special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"])
    bpe_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

    # Обучаем на всем (или части) очищенном датасете
    bpe_tokenizer.train_from_iterator(texts, trainer)

    encoded = bpe_tokenizer.encode(sample_text)
    print(f"BPE Vocab Size: {bpe_tokenizer.get_vocab_size()}")
    print(f"BPE Encoded Sequence Length: {len(encoded.ids)}")
    return bpe_tokenizer

bpe_tokenizer = run_tokenization_tasks(dm.final_data)


--- Tokenization Tasks ---
Char Vocab Size: 1135
Sample Char Sequence Length: 5284
Word Vocab Size (subset): 7787
Sample Word Sequence Length: 791
BPE Vocab Size: 5000
BPE Encoded Sequence Length: 1350


In [20]:
from datasets import load_dataset
import torch

class WikiTextProcessing(CommonCrawlDataModule):
    def __init__(self, cc_bpe_tokenizer, **kwargs):
        # Наследуем инициализацию, но передаем уже обученный BPE токенизатор
        super().__init__(warc_url=None, **kwargs)
        self.bpe_tokenizer = cc_bpe_tokenizer # Используем токенизатор из CC

    def fetch_wikitext(self):
        print("Loading WikiText from Hugging Face...")
        # Загружаем wikitext-2 (он компактный)
        dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
        # Извлекаем тексты, убирая пустые строки, которые часто встречаются в raw wikitext
        return [line['text'] for line in dataset if len(line['text'].strip()) > 0]

    def process_wikitext(self):
        raw_texts = self.fetch_wikitext()
        processed_data = []

        print("Cleaning and calculating metrics for WikiText...")
        for raw_text in raw_texts:
            cleaned = self.clean_text(raw_text)
            if cleaned:
                # В WikiText сегментация может не понадобиться, если строки короткие,
                # но мы прогоним через расчет метрик
                entropy, density = self.calculate_metrics(cleaned)
                processed_data.append(cleaned)

        print(f"WikiText processed. Remaining objects: {len(processed_data)}")
        return processed_data

    ##  PACKED BATCHING
    def create_packed_batches(self, texts, block_size=512):
        print(f"Starting Packed Batching (block size: {block_size})...")

        # 1. Токенизируем все тексты и добавляем EOS токен в конец каждого
        all_token_ids = []
        eos_token_id = self.bpe_tokenizer.token_to_id("[SEP]") # Или другой разделитель
        if eos_token_id is None:
            eos_token_id = 0

        for text in texts:
            encoded = self.bpe_tokenizer.encode(text)
            all_token_ids.extend(encoded.ids + [eos_token_id])

        # 2. Разбиваем длинный список токенов на блоки фиксированной длины
        total_length = len(all_token_ids)
        # Отрезаем остаток, который не влезает в полный блок
        total_length = (total_length // block_size) * block_size

        packed_batches = []
        for i in range(0, total_length, block_size):
            batch = all_token_ids[i : i + block_size]
            packed_batches.append(torch.tensor(batch))

        print(f"Created {len(packed_batches)} packed blocks.")
        return packed_batches

# 1. Используем bpe_tokenizer, который был обучен в предыдущем задании
wiki_processor = WikiTextProcessing(cc_bpe_tokenizer=bpe_tokenizer)

# 2. Обработка
wiki_texts = wiki_processor.process_wikitext()

# 3. Упаковка в батчи
packed_data = wiki_processor.create_packed_batches(wiki_texts, block_size=512)

# Проверка случайного батча
if len(packed_data) > 0:
    print(f"Sample packed batch shape: {packed_data[0].shape}")
    print(f"Sample tokens: {packed_data[0][:10]}...")

Using device: cuda


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading WikiText from Hugging Face...


README.md: 0.00B [00:00, ?B/s]

...

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Cleaning and calculating metrics for WikiText...
WikiText processed. Remaining objects: 20871
Starting Packed Batching (block size: 512)...
Created 6175 packed blocks.
Sample packed batch shape: torch.Size([512])
Sample tokens: tensor([4352,   78,  161, 1405,   58, 2299,   93, 1272,   69,   23])...
